# Phase 4d: Step 3 — Spatial Analysis (H3)

**Date:** 2026-11-03  
**Driver:** `code/R/04d_step3_spatial.R --date 2026-11-03`  
**Results:** `results/2026-11-03/h3/`

---

## What is spatial analysis?

H3 asks whether AI data-center concentration clusters spatially and whether neighbor states' AI activity spills over into own renewable outcomes.

**Three models:**

1. **Moran's I** — tests whether the spatial autocorrelation of AI or outcomes is higher than chance. Null: random spatial arrangement.

2. **SLX** (Spatial Lag of X) — the memo's fallback and the clustered-SE companion to the SDM:  
   `Y = b1·AI(t-1) + θ1·W·AI(t-1) + b·X + θ·W·X + μ_i + λ_t`  
   Only clustered-SE model; direct (b1) and neighbor (θ1) effects separable; total = b1+θ1 with delta-method SE.

3. **SDM** (Spatial Durbin Model) — adds spatial lag of Y (ρ·W·Y) to the SLX:  
   `Y = ρ·WY + b1·AI(t-1) + θ1·W·AI(t-1) + b·X + θ·W·X + μ_i + λ_t`  
   Estimated via ML (SDPDmod::SDPDm). The SDM indirect effect is the main H3 test.  
   Direct/indirect/total impacts from `impactsSDPDm()` account for the feedback loop in ρ·WY.

**W matrices:**
- Main: **invdist-50** (`W_inverse_distance_rowstd.csv`, all 50 states, row-standardized)
- Checks: **invdist-48** (drop AK+HI, re-normalized) and **border-48** (contiguity, row-standardized)

**Controls:** ctrl4 = ln_gdp_pc + manuf_share + price_c_kwh + pop_growth.  
**SLX SEs:** clustered by state (CRV1). **SDM SEs:** asymptotic ML (not clustered).  
**Sample:** 50 states × 11 years = 550 obs (invdist-50); 48 states × 11 years = 528 (48-state W checks).

**SDPDm notes:**
- `LYtrans=TRUE` applies Lee-Yu (2010) bias correction for two-way FE panel (main result when converged).
- `LYtrans=FALSE` = uncorrected ML (shown beside; used for splm cross-check).
- `demn=FALSE`: `effect='twoways'` handles FE removal internally; demn is for explicit Lee-Yu demeaning (not needed here).
- **Convergence flag:** SDPDmod LYtrans=TRUE gave rho > 1 for ln_ec and ln_ee with invdist W (rho ≈ 4.8–6.1). These are outside the valid interval (1/λ_min, 1) = (−4.74, 1) for invdist-50 and indicate a numerical convergence failure. Use LYtrans=FALSE for those outcomes. renew_share converges correctly (rho=0.776).
- SEs are asymptotic ML; compare b1 and θ1 with SLX (which has clustered SEs).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

DATE = '2026-11-03'
H3   = f'../results/{DATE}/h3/'
FIGS = f'../results/{DATE}/figures/'

moran  = pd.read_csv(H3 + 'h3_moran.csv')
lm     = pd.read_csv(H3 + 'h3_lm_tests.csv')
slx    = pd.read_csv(H3 + 'h3_slx.csv')
sdm_c  = pd.read_csv(H3 + 'h3_sdm_core.csv')
sdm_i  = pd.read_csv(H3 + 'h3_sdm_impacts.csv')
splm_c = pd.read_csv(H3 + 'h3_splm_check.csv')
print('Moran rows:', len(moran))
print('SLX rows:', len(slx))
print('SDM core rows:', len(sdm_c))
print('SDM impacts rows:', len(sdm_i))

---
## 1. Moran's I — spatial clustering (H3a)

Null: random spatial arrangement. `alternative='greater'`: tests for positive spatial autocorrelation.  
999 permutations, p-value = proportion of simulated I ≥ observed I.

In [ ]:
# renew_share by year, invdist-50
rs_i50 = moran[(moran.variable=='renew_share') & (moran.W_version=='invdist50') & (moran.type=='raw')]
rs_i50 = rs_i50[['year','moran_I','p_value']].sort_values('year')
rs_i50.columns = ['Year','Moran I','p']
rs_i50 = rs_i50.round({'Moran I':4, 'p':3})
print('renew_share Moran I, invdist-50:')
display(rs_i50.set_index('Year'))

In [ ]:
# AI lag1 Moran I, all W
ai_m = moran[(moran.variable=='ai_dc_per1000_lag1') & (moran.type=='raw')]
ai_piv = ai_m.pivot_table(index='year', columns='W_version', values='moran_I').round(4)
print('AI lag1 Moran I by year and W:')
display(ai_piv)

In [ ]:
# Residual Moran I (renew_share TWFE residuals)
res_m = moran[(moran.variable=='resid_renew_share') & (moran.type=='resid')]
res_piv = res_m.pivot_table(index='year', columns='W_version', values=['moran_I','p_value']).round(4)
print('Residual Moran I (renew_share TWFE residuals):')
display(res_piv)

In [ ]:
Image(FIGS + 'h3_moran_time_renew_share.png')

### Reading (Moran's I)

**renew_share clustering is significant in every year (p ≤ 0.002, invdist-50)**, rising from I=0.094 in 2015 to I=0.157 in 2025. States with high renewable shares cluster near each other (New England, PNW, Mountain West) and low-share states cluster in the Southeast — this trend is strengthening over time.

**AI lag1 is also positively clustered** (invdist-50), with I peaking in 2021–2022 before stabilizing. This provides motivation for a spatial model.

**Residual Moran's I:** After removing the TWFE + AI(t-1) signal, the remaining spatial autocorrelation in renew_share is small and mostly p > 0.05, suggesting the TWFE model adequately absorbs spatial patterns in the error. The SDM adds a spatial lag of Y to test for spillover beyond what the TWFE captures.

---
## 2. LM pre-tests (optional)

LM-lag (lml), LM-error (lme), and robust versions. Two-way FE model, invdist-50 listw.

In [ ]:
lm_piv = lm.pivot_table(index='outcome', columns='test', values=['statistic','p_value'])
lm_piv = lm_piv.round(3)
print('LM pre-tests (invdist-50):')
display(lm_piv)

### Reading (LM tests)

- **ln_ec, renew_share:** All four LM tests fail to reject the null (p > 0.1). No strong evidence of spatial lag or error dependence beyond two-way FE.
- **ln_ee:** LM-error (lme p=0.005) and robust versions (rlml p<0.001, rlme p<0.001) reject the null. This suggests spatial error correlation in the efficiency equation. The SDM spatial lag absorbs some of this; a spatial error model is an alternative not pursued here.
- **Implication:** SDM is estimated for all three outcomes per the methods memo; the LM tests are informational only.

---
## 3. SLX — memo fallback, clustered-SE companion

**Model:** `Y = b1·AI(t-1) + θ1·W·AI(t-1) + b·X + θ·W·X + μ_i + λ_t`  
**SE:** clustered by state (CRV1). **Total = b1+θ1** with delta-method clustered SE.  
No spatial lag of Y → no feedback loop, no ML estimation. The only spatial model with clustered SEs.

In [ ]:
slx_show = slx[['outcome','W_version','b1','se_b1','p_b1','theta1','se_theta1','p_theta1',
                'total','se_total','p_total','N']].copy()
for c in ['b1','se_b1','theta1','se_theta1','total','se_total']:
    slx_show[c] = slx_show[c].round(5)
for c in ['p_b1','p_theta1','p_total']:
    slx_show[c] = slx_show[c].round(3)
display(slx_show.set_index(['outcome','W_version']))

In [ ]:
Image(FIGS + 'h3_slx_coefplot.png')

### Reading (SLX)

- **ln_ec, ln_ee:** All coefficients (b1, θ1, total) are null across all three W specifications (p > 0.5). No evidence of direct or spillover AI effects on consumption or efficiency.
- **renew_share:** b1 is significant across all three W specs (invdist-50: b1=−0.0136, p=0.014; invdist-48: b1=−0.0136, p=0.014; border-48: b1=−0.0135, p=0.025). θ1 is negative but insignificant (p=0.108–0.117). **Total (b1+θ1) is borderline** (invdist-50: total=−0.069, p=0.068; border-48: total=−0.043, p=0.057) — marginally significant at 10%, not at 5%.
- The direct effect b1 is consistent with H1 baseline (−0.0101 to −0.0136). The neighbor spillover θ1 is negative (high-AI neighbors reduce own renewable share) but imprecise.
- Compare with SDM: SLX b1 ≈ SDM b1 (−0.0136 vs −0.0124); θ1 more negative in SLX (−0.055 vs −0.046). The SDM absorbs some of the θ1 signal into the ρ·WY term.

---
## 4. SDM — main H3 test (ML SEs)

**SDPDmod::SDPDm**, `model='sdm'`, `effect='twoways'`, `dynamic=FALSE`.  
LYtrans=TRUE (Lee-Yu bias correction) is the main result where valid. LYtrans=FALSE shown beside for splm cross-check.

**SDPDm notes:**
- `LYtrans=TRUE` applies the Lee-Yu (2010) transformation to correct finite-T bias in the ML rho estimate.
- `demn=FALSE`: two-way FE are absorbed via `effect='twoways'` in the ML objective; demn would apply explicit Lee-Yu demeaning as an alternative.
- **Coefficient names:** main effects use original variable names; W-lagged terms use `W*` prefix (e.g. `W*ai_dc_per1000_lag1`).
- **Impacts:** `impactsSDPDm(NSIM=500)` Monte Carlo from asymptotic distribution. For near-zero ρ, SE inflates sharply; use the analytical formula (b1+θ1)/(1−ρ) as a cross-check.

**Convergence flag (LYtrans=TRUE):** SDPDmod gave ρ>1 for ln_ec and ln_ee with invdist W (ρ≈4.8–6.1), outside the valid interval (−4.74, 1). This is a numerical convergence failure, not a substantive finding. Use LYtrans=FALSE for those outcomes.

In [ ]:
# SDM coefficients — main W (invdist-50), LYtrans=TRUE and FALSE
sdm_main = sdm_c[sdm_c.W_version=='invdist50'][['outcome','LYtrans','rho','rho_valid','b1','theta1','analytical_total']]
sdm_main = sdm_main.round({'rho':4,'b1':6,'theta1':6,'analytical_total':6})
print('SDM coefficients — invdist-50:')
display(sdm_main.set_index(['outcome','LYtrans']))

In [ ]:
# SDM impacts — main W, label validity
sdm_imp_main = sdm_i[sdm_i.W_version=='invdist50'][[
    'outcome','LYtrans','rho_valid',
    'direct','sd_direct','p_direct',
    'indirect','sd_indirect','p_indirect',
    'total','sd_total','p_total']].copy()
for c in ['direct','sd_direct','indirect','sd_indirect','total','sd_total']:
    sdm_imp_main[c] = sdm_imp_main[c].round(5)
for c in ['p_direct','p_indirect','p_total']:
    sdm_imp_main[c] = sdm_imp_main[c].round(3)
print('SDM AI impacts — invdist-50 (rho_valid flags convergence):')
display(sdm_imp_main.set_index(['outcome','LYtrans']))

In [ ]:
Image(FIGS + 'h3_sdm_impacts_bar.png')

### Reading (SDM — main W invdist-50)

**renew_share (H3 primary result):**
- LYtrans=TRUE (valid): ρ=0.776, b1=−0.0124, θ1=−0.046
  - Direct impact: **−0.017 (p=0.001)** — robust negative own-state effect
  - Indirect impact: **−0.252 (p=0.063)** — large negative neighbor spillover, borderline 10%
  - Total: **−0.269 (p=0.054)** — driven mostly by indirect (neighbor) channel
- LYtrans=FALSE: ρ=0.617, direct=−0.015 (p<0.001), indirect=−0.165 (p=0.038), total=−0.181 (p=0.029)
- **Interpretation:** Strong spatial dependence (ρ≈0.6–0.8). AI data-center concentration is associated with less renewable share both directly and in neighbors — consistent with AI-driven demand growth across interconnected regional grids.

**ln_ec, ln_ee:**
- LYtrans=TRUE: **convergence failure** (ρ>1 for both outcomes with invdist W). Use LYtrans=FALSE.
- LYtrans=FALSE ln_ec: ρ=0.137, direct=−0.002 (p=0.878), total≈0.040 (p=0.945). Null.
- LYtrans=FALSE ln_ee: ρ=−0.018 ≈ 0, direct=0.004 (p=0.678), total=0.022 (p=0.965). Null. (Near-zero ρ → impactsSDPDm SE=0.5, dominated by simulation noise; analytical formula gives −0.003.)
- Consistent with H1 TWFE: no AI effect on consumption or efficiency.

**SDM vs SLX comparison (renew_share, invdist-50):**
- b1: SLX=−0.0136, SDM(LYF)=−0.0128 — close
- θ1: SLX=−0.055, SDM(LYF)=−0.055 — essentially identical
- SDM accounts for ρ·WY feedback, amplifying the total to −0.181 (vs SLX total=−0.069)

---
## 5. SDM sensitivity (invdist-48, border-48, all 3 outcomes)

In [ ]:
sdm_sens = sdm_i[sdm_i.W_version != 'invdist50'][[
    'outcome','W_version','rho_valid',
    'direct','p_direct','indirect','p_indirect','total','p_total']].copy()
for c in ['direct','indirect','total']:
    sdm_sens[c] = sdm_sens[c].round(5)
for c in ['p_direct','p_indirect','p_total']:
    sdm_sens[c] = sdm_sens[c].round(3)
print('SDM sensitivity (LYtrans=TRUE) — invalid rows = convergence failure:')
display(sdm_sens.set_index(['outcome','W_version']))

### Reading (sensitivity)

- **renew_share:** Consistent negative effects across all W. border-48 direct=−0.0097 (p=0.014), indirect=−0.031 (p=0.041), total=−0.041 (p=0.024). invdist-48 also negative (total=−0.265, p=0.057). The main result is robust to W choice.
- **ln_ec:** LYtrans=TRUE convergence failure for both invdist-48 (ρ=4.77) and border-48 (ρ=1.48). Discard.
- **ln_ee:** invdist-48 convergence failure (ρ=6.24). **border-48 converges:** direct=0.011 (p=0.005), indirect=0.034 (p=0.008), total=0.045 (p=0.004). States with more AI neighbors have higher energy efficiency in border W — spatially transmitted efficiency gains, possibly through shared grid infrastructure. This is a single sensitivity result; not in the main analysis.

---
## 6. splm cross-check

**Compare ρ, b1, θ1 from splm vs SDPDm LYtrans=FALSE** (same no-correction baseline).  
splm fits: `spml(Y ~ AI + W·AI + ctrl4 + W·ctrl4, lag=TRUE, spatial.error='none', effect='twoways')`  
Hand-made W·X columns; ρ from `m$spat.coef`; b1/θ1 from `coef()`. **Do not take impacts from splm** — W·AI is a fixed regressor, not inside splm's impact machinery.

In [ ]:
sdm0 = sdm_c[(sdm_c.W_version=='invdist50') & (~sdm_c.LYtrans)][['outcome','rho','b1','theta1']]
sdm0.columns = ['outcome','rho_sdm','b1_sdm','theta1_sdm']
sp0  = splm_c[['outcome','rho','b1','theta1']]
sp0.columns  = ['outcome','rho_spl','b1_spl','theta1_spl']
comp = sdm0.merge(sp0, on='outcome')
for c in comp.columns[1:]:
    comp[c] = comp[c].round(5)
print('SDPDm LYtrans=FALSE vs splm (invdist-50):')
display(comp.set_index('outcome'))

### Reading (cross-check)

**b1 and θ1 match to 5 decimal places** between SDPDm (LYtrans=FALSE) and splm — confirming correct model specification and hand-made W·X columns. The only difference is ρ: SDPDm 0.137 vs splm 0.137 (ln_ec), −0.018 vs −0.019 (ln_ee), 0.617 vs 0.622 (renew_share). The small ρ differences are consistent with different ML approximations. Cross-check passes.

---
## 7. Maps

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
vars_map = ['renew_share','ai_dc_per1000_lag1','ln_ec','ln_ee']
titles   = ['Renewable Share','AI DC/1k workers','ln(Elec. Consump.)','ln(Energy Effic.)']
for col, (vn, ttl) in enumerate(zip(vars_map, titles)):
    for row, yr in enumerate([2015, 2025]):
        fn = FIGS + f'h3_map_{vn.replace("/","_")}_{yr}.png'
        img = plt.imread(fn)
        axes[row,col].imshow(img)
        axes[row,col].axis('off')
        axes[row,col].set_title(f'{ttl}\n{yr}', fontsize=9)
plt.tight_layout()
plt.show()

---
## Summary (H3 — Spatial)

| Test | Outcome | Key result | H3 verdict |
|---|---|---|---|
| Moran's I | renew_share | I=0.094–0.157, p≤0.002 all years | Significant spatial clustering |
| Moran's I | AI lag1 | Positive, rising to 2022 | Spatial AI clustering confirmed |
| Residual Moran's I | renew_share | Small, mostly p>0.05 after TWFE | TWFE absorbs most spatial error |
| SLX | renew_share | b1=−0.014(p=0.014), θ1=−0.055(p=0.117), total=−0.069(p=0.068) | Direct effect sig; neighbor borderline |
| SLX | ln_ec, ln_ee | All null | — |
| SDM (invdist-50, LYtrans=TRUE) | renew_share | ρ=0.776, direct=−0.017(p=0.001), indirect=−0.252(p=0.063), total=−0.269(p=0.054) | **H3 supported** — large indirect effect |
| SDM (LYtrans=FALSE) | renew_share | ρ=0.617, direct=−0.015(p<0.001), indirect=−0.165(p=0.038), total=−0.181(p=0.029) | Same conclusion, smaller magnitude |
| SDM (LYtrans=TRUE) | ln_ec, ln_ee | ρ>1 (convergence failure) | Discard; use LYtrans=FALSE |
| SDM sensitivity | renew_share | border-48 total=−0.041(p=0.024) | Robust to W choice |
| splm cross-check | all | b1/θ1 match SDPDm to 5 d.p. | Model verified |

**Overall H3 finding:** Renewable share shows significant spatial dependence (ρ≈0.6–0.8). The SDM indirect effect of AI on renew_share is **negative and large** (−0.165 to −0.252), driven by neighbor spillovers through the ρ·WY channel. High-AI states in a region appear to draw renewable capacity into their own state while suppressing shares in neighboring states, possibly via interstate grid competition for PPAs and renewable developers. This is exploratory — the indirect effect is borderline (p=0.038–0.063) and sensitive to LYtrans choice.

**Next step:** Phase 4e — robustness (LBNL RPS control, t-2/t-3 lags, alternative AI measure).